# 1. Introdução ao Estudo

Este notebook tem como objetivo realizar uma análise exploratória completa de um dataset de e-commerce
de moda e lifestyle, explorando padrões de preço, desconto, avaliação de usuários, marcas e categorias.

Mesmo sem estar associado a uma empresa real, o estudo assume a perspectiva de um analista de dados
avaliando estratégias de precificação, percepção de valor e desempenho de portfólio.

Perguntas centrais:
- Como preços e descontos se distribuem?
- Desconto está associado a melhor avaliação?
- Quais marcas e categorias entregam maior valor percebido?
- Existem produtos com preços inflacionados ou subvalorizados?


In [ ]:
# 2. Importação de bibliotecas e carregamento dos dados

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")

df = pd.read_csv("./dados/data.csv")
df.head()


# 3. Visão Geral da Base de Dados

Nesta seção:
- Observamos o formato da base
- Tipos de dados
- Quantidade de registros
- Primeira avaliação estrutural


In [ ]:
df.info()

In [ ]:
df.describe()

# 4. Qualidade dos Dados

Antes de qualquer insight, precisamos entender:
- Valores nulos
- Possíveis inconsistências
- Colunas que exigem tratamento

Hipótese:
A base pode conter valores ausentes ou distorções típicas de dados coletados via scraping.


In [ ]:
df.isnull().sum().sort_values(ascending=False)

# 5. Análise de Preços e Descontos

Aqui investigamos:
- Distribuição de preços originais e com desconto
- Intensidade dos descontos
- Diferença absoluta e percentual entre preços

Hipóteses:
- A maioria dos produtos apresenta algum nível de desconto
- Existem preços âncora artificialmente inflados


In [ ]:
plt.figure()
sns.histplot(df["marked_price"], bins=50, kde=True)
plt.title("Distribuição do Preço Original")
plt.show()

In [ ]:
plt.figure()
sns.histplot(df["discounted_price"], bins=50, kde=True)
plt.title("Distribuição do Preço com Desconto")
plt.show()

In [ ]:
plt.figure()
sns.histplot(df["discount_percent"], bins=50, kde=True)
plt.title("Distribuição do Percentual de Desconto")
plt.show()

# 6. Relação entre Desconto e Avaliação

Pergunta-chave:
Produtos mais descontados são melhor avaliados?

Hipótese:
Desconto agressivo não garante melhor percepção de qualidade.

In [ ]:
plt.figure()
sns.scatterplot(
    data=df,
    x="discount_percent",
    y="rating",
    alpha=0.4
)
plt.title("Desconto (%) vs Rating")
plt.show()

# 7. Análise de Avaliações (Rating e Rating Count)

Nesta etapa:
- Avaliamos a distribuição de ratings
- Identificamos possíveis vieses
- Combinamos rating e volume de avaliações

Hipótese:
Produtos com muitas avaliações tendem a ter ratings mais estáveis.


In [ ]:
plt.figure()
sns.histplot(df["rating"], bins=20, kde=True)
plt.title("Distribuição de Ratings")
plt.show()

In [ ]:
plt.figure()
sns.histplot(df["rating_count"], bins=50, kde=True)
plt.title("Distribuição do Número de Avaliações")
plt.show()

# 8. Preço vs Avaliação

Aqui buscamos responder:
Produtos mais caros são melhor avaliados?

Hipótese:
Preço elevado só se sustenta com alta percepção de valor.

In [ ]:
plt.figure()
sns.scatterplot(
    data=df,
    x="discounted_price",
    y="rating",
    alpha=0.4
)
plt.title("Preço com Desconto vs Rating")
plt.show()

# 9. Análise por Marca

Exploramos:
- Quantidade de produtos por marca
- Preço médio
- Rating médio simples
- Rating médio ponderado pelo número de avaliações

Hipótese:
Nem toda marca premium entrega avaliação premium.


In [ ]:
brand_summary = (
    df.groupby("brand_name")
    .agg(
        produtos=("product_name", "count"),
        preco_medio=("discounted_price", "mean"),
        rating_medio=("rating", "mean"),
        rating_ponderado=("rating", lambda x: np.average(x, weights=df.loc[x.index, "rating_count"]))
    )
    .sort_values("produtos", ascending=False)
)

brand_summary.head(10)


# 10. Análise por Categoria de Produto

Objetivos:
- Identificar categorias mais caras
- Categorias mais descontadas
- Categorias melhor avaliadas

Hipótese:
Algumas categorias dependem estruturalmente de desconto.


In [ ]:
category_summary = (
    df.groupby("product_tag")
    .agg(
        produtos=("product_name", "count"),
        preco_medio=("discounted_price", "mean"),
        desconto_medio=("discount_percent", "mean"),
        rating_medio=("rating", "mean")
    )
    .sort_values("produtos", ascending=False)
)

category_summary.head(10)


# 11. Criação de KPIs Globais

Nesta seção criamos KPIs que serão replicáveis no Power BI:
- Preço médio
- Desconto médio
- Rating médio simples e ponderado
- Proporção de produtos bem avaliados


In [ ]:
kpis = {
    "Preço Médio Original": df["marked_price"].mean(),
    "Preço Médio com Desconto": df["discounted_price"].mean(),
    "Desconto Médio (%)": df["discount_percent"].mean(),
    "Rating Médio": df["rating"].mean(),
    "Rating Médio Ponderado": np.average(df["rating"], weights=df["rating_count"]),
    "% Produtos Rating >= 4.5": (df["rating"] >= 4.5).mean() * 100
}

pd.Series(kpis)


# 12. Métrica de Valor Percebido (Feature Engineering)

Criamos um índice próprio para avaliar custo-benefício:

Valor Percebido = (rating × log(rating_count + 1)) / preço com desconto

Essa métrica favorece produtos:
- Bem avaliados
- Com muitas avaliações
- Com preço acessível


In [ ]:
df["valor_percebido"] = (
    df["rating"] * np.log1p(df["rating_count"])
) / df["discounted_price"]

df.sort_values("valor_percebido", ascending=False).head(10)


# 13. Produtos Supervalorizados vs Subvalorizados

Aqui observamos extremos:
- Produtos caros e mal avaliados
- Produtos baratos e muito bem avaliados

Esses casos são especialmente relevantes para decisões de portfólio.


In [ ]:
df["flag_supervalorizado"] = (df["discounted_price"] > df["discounted_price"].quantile(0.75)) & (df["rating"] < 3.5)
df["flag_subvalorizado"] = (df["discounted_price"] < df["discounted_price"].quantile(0.25)) & (df["rating"] >= 4.5)

df[["product_name", "discounted_price", "rating", "flag_supervalorizado", "flag_subvalorizado"]].head(10)


# 14. Conclusões Analíticas

Principais aprendizados possíveis a partir deste estudo:
- Desconto não garante boa avaliação
- Preço elevado exige percepção de qualidade
- Algumas marcas e categorias entregam melhor valor percebido
- Métricas derivadas ampliam muito a capacidade analítica do dashboard

Este notebook serve como base direta para:
- Criação de medidas DAX no Power BI
- Construção de dashboards executivos
- Discussões estratégicas sobre precificação e portfólio
